# A4 欧氏 GCN 与 A5 类别平衡 GCN

独立 kernel、单卡顺序运行两组；默认 3 个切片 × 5 个 seed。完整协议见 [README](README.md#a0a5-欧氏对照实验)。本轮是探索性表示实验，不使用隐藏切片选参。


In [1]:
# 必须在导入 torch 前运行；切换显卡或更新代码后重启 kernel。
import os
import sys
GPU_ID = "0"  # 调度器管理可见卡时设为 None。
if "torch" in sys.modules:
    raise RuntimeError("请重启独立 kernel，并从第一格运行。")
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "scheduler/default"))


CUDA_VISIBLE_DEVICES = 0


In [2]:
from pathlib import Path
from copy import deepcopy
PROJECT = next((candidate for parent in (Path.cwd(), *Path.cwd().parents)
                for candidate in (parent, parent / "MMDeepSpatial")
                if (candidate / "mmdeepspatial/euclidean.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("请在 MMDeepSpatial 或其父目录打开 notebook。")
sys.path.insert(0, str(PROJECT))
import pandas as pd
from mmdeepspatial.euclidean import EUCLIDEAN_CONFIG, run_arm, compare_arms
from mmdeepspatial.data import input_paths
print("Project:", PROJECT)


Project: /3250608056/code/DeepSpatial/MMDeepSpatial


## 统一配置

三个 notebook 的科学配置须一致。A3/A5 使用训练标签计算重建损失权重，probe 设置不变。GCN 只用可见同切片表达构图，新细胞仅连接冻结训练参考库，不使用隐藏 XY 或查询之间的边。

若做短流程检查，统一改成一个 holdout、一个 seed、两轮训练，并将 `REUSE_A2_ROOT=None`，另用测试输出目录。正式运行恢复全部配置。


In [3]:
ARMS_TO_RUN = ['A4', 'A5']
DATA_GLOB = str(PROJECT.parent / "data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad")
OUTPUT_ROOT = PROJECT / "outputs/euclidean_v1"
HOLDOUTS = [1, 2, 3]
SEEDS = [42, 0, 1, 2, 3]
EPOCHS = 100
DIM = 32
HIDDEN = 128
GRAPH_K = 10
BALANCE_POWER = 0.5  # 逆平方根频率，样本权重全训练集均值为 1。
LABEL_KEY = "cell_class"
LAYER = None
TRANSFORM = "log1p"  # X 已 log / 标准化时统一改 identity。
DEVICE = "cuda"
# 可复用上一轮兼容的 E0 AE；设为 None 则重新训练 A2。
REUSE_A2_ROOT = PROJECT / "outputs/representations_v2/E0_ae"
if not REUSE_A2_ROOT.is_dir():
    REUSE_A2_ROOT = None
CONFIG = deepcopy(EUCLIDEAN_CONFIG)
CONFIG.update(input_glob=DATA_GLOB, output_root=str(OUTPUT_ROOT), holdouts=HOLDOUTS,
              seeds=SEEDS, epochs=EPOCHS, dim=DIM, hidden=HIDDEN, graph_k=GRAPH_K,
              balance_power=BALANCE_POWER, label_key=LABEL_KEY, layer=LAYER,
              transform=TRANSFORM, device=DEVICE,
              reuse_a2_root=str(REUSE_A2_ROOT) if REUSE_A2_ROOT else None)
CONFIG


{'input_glob': '/3250608056/code/DeepSpatial/data/merfish_mouse_hypothalamus/**/merfish_[0-4].h5ad',
 'output_root': '/3250608056/code/DeepSpatial/MMDeepSpatial/outputs/euclidean_v1',
 'holdouts': [1, 2, 3],
 'seeds': [42, 0, 1, 2, 3],
 'label_key': 'cell_class',
 'layer': None,
 'transform': 'log1p',
 'n_genes': 0,
 'rare_fraction': 0.05,
 'dim': 32,
 'hidden': 128,
 'epochs': 100,
 'batch_size': 512,
 'lr': 0.001,
 'weight_decay': 1e-05,
 'grad_clip': 5.0,
 'curvature': 1.0,
 'fusion_alpha': 0.5,
 'log_every': 10,
 'e0_method': 'ae',
 'device': 'cuda',
 'cpu_threads': 4,
 'retrieval_ks': [1, 5, 15],
 'query_chunk': 128,
 'reference_chunk': 2048,
 'coverage_quantile': 0.95,
 'coverage_calibration_max': 2048,
 'probe_c': 1.0,
 'probe_class_weight': 'balanced',
 'probe_max_iter': 2000,
 'shuffled_probe': False,
 'max_tangent_radius': 1.5,
 'diagnostics': False,
 'diagnostic_alphas': [0.25, 0.75],
 'graph_k': 10,
 'graph_query_chunk': 512,
 'balance_power': 0.5,
 'reuse_a2_root': '/32506

In [4]:
paths = input_paths(CONFIG["input_glob"])
display(pd.DataFrame({"slice_index": range(len(paths)), "file": [str(p) for p in paths],
                      "held_out": [i in HOLDOUTS for i in range(len(paths))]}))
print("Arms:", ARMS_TO_RUN, "runs per arm:", len(HOLDOUTS)*len(SEEDS))
print("A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。")


,slice_index,file,held_out
0,0,/3250608056/code/DeepSpatial/data/merfish_mous...,False
1,1,/3250608056/code/DeepSpatial/data/merfish_mous...,True
2,2,/3250608056/code/DeepSpatial/data/merfish_mous...,True
3,3,/3250608056/code/DeepSpatial/data/merfish_mous...,True
4,4,/3250608056/code/DeepSpatial/data/merfish_mous...,False


Arms: ['A4', 'A5'] runs per arm: 15
A0/A1 不训练神经编码器；A2 可复用已核验的旧权重。


In [5]:
results = pd.concat([run_arm(arm, CONFIG) for arm in ARMS_TO_RUN], ignore_index=True)
display(results[["arm", "holdout", "seed", "encoder_uses_labels", "encoder_reused",
                 "rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
                 "rare_macro_f1", "rare_macro_retrieval_hit_at_5", "common_macro_f1",
                 "trainable_parameters"]])


Start A4 holdout=1 seed=42 train=(22760, 155) method=gcn balanced=False
E0 seed=42 epoch=1/100 mse=0.87647 boundary=0.000%
E0 seed=42 epoch=10/100 mse=0.60780 boundary=0.000%
E0 seed=42 epoch=20/100 mse=0.58062 boundary=0.000%
E0 seed=42 epoch=30/100 mse=0.56620 boundary=0.000%
E0 seed=42 epoch=40/100 mse=0.55722 boundary=0.000%
E0 seed=42 epoch=50/100 mse=0.55145 boundary=0.000%
E0 seed=42 epoch=60/100 mse=0.54713 boundary=0.000%
E0 seed=42 epoch=70/100 mse=0.54373 boundary=0.000%
E0 seed=42 epoch=80/100 mse=0.54087 boundary=0.000%
E0 seed=42 epoch=90/100 mse=0.53844 boundary=0.000%
E0 seed=42 epoch=100/100 mse=0.53653 boundary=0.000%
 rare_macro_auprc  ultra_rare_macro_auprc  rare_macro_f1  overall_accuracy
         0.685522                0.479601       0.600026          0.874033
Start A4 holdout=1 seed=0 train=(22760, 155) method=gcn balanced=False
E0 seed=0 epoch=1/100 mse=0.88650 boundary=0.000%
E0 seed=0 epoch=10/100 mse=0.61000 boundary=0.000%
E0 seed=0 epoch=20/100 mse=0.58007

,arm,holdout,seed,encoder_uses_labels,encoder_reused,rare_macro_auprc,ultra_rare_macro_auprc,low_rare_macro_auprc,rare_macro_f1,rare_macro_retrieval_hit_at_5,common_macro_f1,trainable_parameters
0,A4,1,42,False,False,0.685522,0.479601,0.932627,0.600026,0.756679,0.915100,48315
1,A4,1,0,False,False,0.700978,0.510114,0.930014,0.592018,0.748244,0.921022,48315
2,A4,1,1,False,False,0.707615,0.514856,0.938925,0.612501,0.788112,0.921529,48315
3,A4,1,2,False,False,0.681978,0.472488,0.933366,0.605830,0.760412,0.921192,48315
4,A4,1,3,False,False,0.691730,0.494876,0.927954,0.601860,0.727754,0.917512,48315
5,A4,2,42,False,False,0.700204,0.519075,0.917559,0.553566,0.769972,0.916813,48315
6,A4,2,0,False,False,0.703667,0.520182,0.923849,0.587725,0.770457,0.917956,48315
7,A4,2,1,False,False,0.709360,0.527495,0.927597,0.575431,0.771529,0.917515,48315
8,A4,2,2,False,False,0.704803,0.518821,0.927980,0.586841,0.768443,0.916700,48315
9,A4,2,3,False,False,0.696279,0.501487,0.930029,0.577260,0.767410,0.919850,48315


In [6]:
metrics = ["rare_macro_auprc", "ultra_rare_macro_auprc", "low_rare_macro_auprc",
           "rare_macro_f1", "rare_macro_recall", "rare_macro_retrieval_hit_at_5",
           "rare_macro_knn_purity_at_15", "common_macro_f1"]
by_fold = results.groupby(["arm", "holdout"])[metrics].mean()
display(by_fold)
# 描述性切片间统计，不是独立生物学重复的置信区间。
display(by_fold.groupby("arm").agg(["mean", "std"]))


rare_macro_auprc  ultra_rare_macro_auprc  low_rare_macro_auprc  \
arm holdout                                                                   
A4  1                0.693565                0.494387              0.932577   
    2                0.702863                0.517412              0.925403   
    3                0.702682                0.512703              0.930657   
A5  1                0.715208                0.528831              0.938861   
    2                0.762608                0.622766              0.930420   
    3                0.751103                0.591042              0.943176   

             rare_macro_f1  rare_macro_recall  rare_macro_retrieval_hit_at_5  \
arm holdout                                                                    
A4  1             0.602447           0.721396                       0.756240   
    2             0.576165           0.747573                       0.769562   
    3             0.602627           0.758300                       0.765617   
A5  1             0.622512           0.754096                       0.772825   
    2             0.589344           0.782439                       0.795990   
    3             0.623148           0.788275                       0.775540   

             rare_macro_knn_purity_at_15  common_macro_f1  
arm holdout                                                
A4  1                           0.613763         0.919271  
    2                           0.615330         0.917767  
    3                           0.631937         0.920129  
A5  1                           0.624649         0.926816  
    2                           0.628465         0.920173  
    3                           0.639246         0.932269

rare_macro_auprc           ultra_rare_macro_auprc            \
                mean       std                   mean       std   
arm                                                               
A4          0.699703  0.005317               0.508167  0.012164   
A5          0.742973  0.024724               0.580880  0.047785   

    low_rare_macro_auprc           rare_macro_f1           rare_macro_recall  \
                    mean       std          mean       std              mean   
arm                                                                            
A4              0.929546  0.003714      0.593746  0.015226          0.742423   
A5              0.937486  0.006488      0.611668  0.019335          0.774937   

              rare_macro_retrieval_hit_at_5            \
          std                          mean       std   
arm                                                     
A4   0.018983                      0.763807  0.006843   
A5   0.018283                      0.781452  0.012664   

    rare_macro_knn_purity_at_15           common_macro_f1            
                           mean       std            mean       std  
arm                                                                  
A4                     0.620343  0.010071        0.919056  0.001196  
A5                     0.630787  0.007570        0.926419  0.006058

## 六组全部完成后统一比较

任何一个 notebook 将下方开关改为 True 后运行。比较会核对完整 seed/split、输入、预处理、源码、容量和 A4/A5 图一致性。保存六组原始指标、稀有度分层、逐类表及预定义配对差值，不自动选择最佳模型。


In [8]:
COMPARE_WHEN_READY = True
if COMPARE_WHEN_READY:
    all_runs, aggregate, all_types = compare_arms(OUTPUT_ROOT)
    display(aggregate)
    display(pd.read_csv(OUTPUT_ROOT / "comparison/paired_deltas.csv"))


,arm,all_macro_auprc_mean,all_macro_auprc_std,all_macro_f1_mean,all_macro_f1_std,all_macro_recall_mean,all_macro_recall_std,all_macro_coverage_mean,all_macro_coverage_std,all_macro_retrieval_hit_at_1_mean,...,low_rare_macro_knn_purity_at_15_mean,low_rare_macro_knn_purity_at_15_std,overall_accuracy_mean,overall_accuracy_std,test_reconstruction_mse_mean,test_reconstruction_mse_std,train_seconds_mean,train_seconds_std,peak_train_gpu_mb_mean,peak_train_gpu_mb_std
0,A0,0.839613,0.011949,0.765405,0.012773,0.803315,0.002152,0.951278,0.020673,0.766739,...,0.885557,0.013322,0.926205,0.001034,NaN,NaN,0.007750,0.010313,0.000000,0.000000
1,A1,0.771135,0.007016,0.698087,0.008608,0.805473,0.025882,0.959048,0.006059,0.714901,...,0.885242,0.008912,0.897843,0.005773,0.525143,0.013307,0.689251,0.061143,0.000000,0.000000
2,A2,0.797742,0.034000,0.705017,0.008358,0.831711,0.039115,0.950274,0.018578,0.694924,...,0.836601,0.014445,0.891718,0.007267,0.408359,0.010057,0.000000,0.000000,0.000000,0.000000
3,A3,0.828413,0.023375,0.743885,0.009146,0.841716,0.023018,0.961016,0.009489,0.720087,...,0.831954,0.004505,0.910122,0.000996,0.430857,0.011429,113.940340,72.315985,70.676465,0.042794
4,A4,0.768100,0.004749,0.680495,0.011465,0.785176,0.014207,0.964868,0.023837,0.729836,...,0.872930,0.013620,0.878023,0.003886,0.582756,0.045668,130.974923,116.765838,106.872917,0.087241
5,A5,0.800784,0.018360,0.695602,0.015649,0.811992,0.013408,0.975760,0.008802,0.743431,...,0.873939,0.013968,0.897014,0.008888,0.594279,0.046561,21.509999,0.135708,106.873405,0.087241


,holdout,seed,delta_all_macro_auprc,delta_all_macro_f1,delta_all_macro_recall,delta_all_macro_coverage,delta_all_macro_retrieval_hit_at_1,delta_all_macro_retrieval_hit_at_5,delta_all_macro_retrieval_hit_at_15,delta_all_macro_knn_purity_at_1,...,delta_low_rare_macro_retrieval_hit_at_5,delta_low_rare_macro_knn_purity_at_5,delta_low_rare_macro_retrieval_hit_at_15,delta_low_rare_macro_knn_purity_at_15,delta_overall_accuracy,delta_test_reconstruction_mse,delta_train_seconds,delta_peak_train_gpu_mb,arm,reference
0,1,42,-0.072427,-0.075853,-0.029298,0.025762,-0.056707,-0.018666,-0.008281,-0.056707,...,0.006430,0.012561,0.001555,0.000913,-0.023754,NaN,1.296315,0.000000,A1,A0
1,1,0,-0.072427,-0.075853,-0.029298,0.024991,-0.056707,-0.018666,-0.008281,-0.056707,...,0.006430,0.012561,0.001555,0.000913,-0.023754,NaN,0.993729,0.000000,A1,A0
2,1,1,-0.072427,-0.075853,-0.029298,0.025779,-0.056707,-0.018666,-0.008281,-0.056707,...,0.006430,0.012561,0.001555,0.000913,-0.023754,NaN,0.405992,0.000000,A1,A0
3,1,2,-0.072427,-0.075853,-0.029298,0.025929,-0.056707,-0.018666,-0.008281,-0.056707,...,0.006430,0.012561,0.001555,0.000913,-0.023754,NaN,0.411628,0.000000,A1,A0
4,1,3,-0.072427,-0.075853,-0.029298,0.025737,-0.056707,-0.018666,-0.008281,-0.056707,...,0.006430,0.012561,0.001555,0.000913,-0.023754,NaN,0.398673,0.000000,A1,A0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,3,42,-0.010898,-0.034869,0.001428,0.015512,0.021375,-0.021927,-0.016721,0.021375,...,-0.000090,0.040254,-0.005783,0.054676,-0.010512,0.146477,-28.416852,36.169434,A5,A3
86,3,0,-0.010749,-0.032766,0.004159,0.005606,0.027888,-0.010328,-0.006953,0.027888,...,-0.005895,0.031488,-0.001331,0.050080,-0.012235,0.144284,-46.422683,36.169434,A5,A3
87,3,1,-0.001532,-0.040974,-0.037498,0.017755,0.049490,-0.001850,-0.010872,0.049490,...,0.007241,0.052393,-0.003138,0.062572,-0.002757,0.151024,-43.772366,36.169434,A5,A3
88,3,2,0.004143,-0.027639,0.015713,0.020311,0.028038,0.005526,0.000208,0.028038,...,0.002325,0.044891,-0.014157,0.053815,-0.011546,0.148558,-34.669784,36.169434,A5,A3
